# Rosati 2022 — Quality Control and HLA Inference Validation Against Ground Truth

**Input**: `rosati_master_tcr_hla.tsv.gz` — the unfiltered master table built in `01_rosati_master_table_construction.ipynb`, containing TCRαβ repertoires and HIBAG-imputed HLA genotypes for 192 patients (98 Crohn's Disease, 94 Healthy) from Elisa Rosati's original data.

**Purpose of this notebook**: two independent goals.

1. **Produce a clean, analysis-ready table** — apply standard TCR quality filters and present the HIBAG HLA genotype in a clinically readable format (one column per HLA locus) instead of raw 0/1/2 dosage across 272 allele columns.
2. **Validate blind HLA inference against this ground truth** — HLAGuessr and THNet were previously run independently on the matched ENA/MiXCR-processed repertoire (PRJEB50045) as fully blind inference (no HLA was used as input). This notebook compares each method's predictions against the HIBAG genotypes shared by Elisa Rosati to measure real-world accuracy.

**Note**: HLAGuessr and THNet are *not* re-run here. Their predictions already exist (`hlaguessr_rosati_predictions.csv`, `thnet_rosati_predictions.csv`) from prior blind inference on the ENA-derived repertoire, which was confirmed to be the same underlying sequencing data as Elisa's RDS files (16–20/20 top clonotype overlap per patient, see notebook 01).

## 0. Setup

In [ ]:
import pandas as pd
import re
import warnings
warnings.filterwarnings('ignore')

DATA    = '/home/immunologylab/bioinformatics/hla-tcr-project/data/'
RESULTS = '/home/immunologylab/bioinformatics/hla-tcr-project/results/'

## Part A — Quality Control

### 1. Load unfiltered master table and inspect CDR3 composition

The master table contains 19,820,069 clonotypes from Elisa Rosati's original TRA/TRB repertoires with no filtering applied. Productive CDR3s should start with cysteine (C) and end with phenylalanine or tryptophan (F/W) — this is checked empirically rather than assumed.

In [ ]:
tcr_cols = ['patient_id','group','chain','v_gene','j_gene','CDR1','CDR2','CDR3','cloneCount','cloneFraction']
master_tcr = pd.read_csv(DATA + 'rosati_master_tcr_hla.tsv.gz', sep='\t', usecols=tcr_cols)

print(f'Total clones: {len(master_tcr):,}')
print(f'Patients: {master_tcr["patient_id"].nunique()}')
print(f'Groups: {master_tcr.groupby("patient_id")["group"].first().value_counts().to_dict()}')

last_chars  = master_tcr['CDR3'].dropna().str[-1].value_counts()
first_chars = master_tcr['CDR3'].dropna().str[0].value_counts()
print(f'\nCDR3 last char (top 3):  {last_chars.head(3).to_dict()}')
print(f'CDR3 first char (top 3): {first_chars.head(3).to_dict()}')

### 2. Apply quality filters

Two filters are applied sequentially:
1. **Productive CDR3** — matches `^C.*[FW]$`. 99.08% of clonotypes pass; the 0.92% removed includes frameshifted sequences (internal stop/underscore artefacts) and non-canonical reading frames.
2. **CDR1/CDR2 present** — CDR1/CDR2 are looked up from germline V-gene dictionaries (notebook 01); a small number of V-genes (e.g. TRAV28, TRBV1, TRBV12-2) are absent from the dictionary and yield null CDR1/CDR2. These rows are dropped.

No minimum-clone-depth filter is needed: even the shallowest patient repertoire exceeds 1,300 clones per chain (see distribution below), well above the conventional 1,000-clone QC threshold used elsewhere in this project.

In [ ]:
productive_mask = master_tcr['CDR3'].str.match(r'^C.*[FW]$', na=False)
master_qc       = master_tcr[productive_mask]
master_clean    = master_qc[master_qc['CDR1'].notna() & master_qc['CDR2'].notna()]

print(f'Original clones:  {len(master_tcr):,}')
print(f'After productive:  {len(master_qc):,}')
print(f'After CDR1/CDR2:   {len(master_clean):,}')
print(f'Total removed:     {len(master_tcr) - len(master_clean):,} '
      f'({(1-len(master_clean)/len(master_tcr))*100:.2f}%)')
print(f'Patients retained: {master_clean["patient_id"].nunique()}/192')

clones_per_pat_chain = master_clean.groupby(['patient_id','chain']).size().unstack(fill_value=0)
MIN_CLONES = 1000
qc_pass = clones_per_pat_chain[
    (clones_per_pat_chain['TRA'] >= MIN_CLONES) & (clones_per_pat_chain['TRB'] >= MIN_CLONES)
].index.tolist()
print(f'\nMinimum clones/patient/chain: TRA={clones_per_pat_chain["TRA"].min()}, '
      f'TRB={clones_per_pat_chain["TRB"].min()}')
print(f'Patients passing ≥{MIN_CLONES} clones/chain: {len(qc_pass)}/192 (all retained)')

### 3. Convert HLA genotype to readable per-locus format

The HIBAG-imputed HLA is stored as dosage (0/1/2) across 272 allele columns — both 2-digit (e.g. `A*01`) and 4-digit (e.g. `A*01:01`) resolutions are present, which is redundant. Only 4-digit alleles are retained, and dosage is collapsed into one human-readable column per locus (`HLA_A`, `HLA_B`, `HLA_C`, `HLA_DRB1`, `HLA_DQB1`, `HLA_DQA1`, `HLA_DPB1`, `HLA_DPA1`), listing all carried alleles (dosage ≥ 1) separated by `/`.

In [ ]:
hla_col_names = pd.read_csv(DATA + 'rosati_master_tcr_hla.tsv.gz', sep='\t', nrows=0).columns
hla_cols      = [c for c in hla_col_names if '*' in str(c)]
four_digit    = [c for c in hla_cols if re.match(r'^[A-Z]+\d*\*\d{2}:\d{2}$', c)]

master_hla = pd.read_csv(DATA + 'rosati_master_tcr_hla.tsv.gz', sep='\t',
                          usecols=['patient_id'] + hla_cols)
hla_per_patient = master_hla.drop_duplicates('patient_id').reset_index(drop=True)

loci = ['A','B','C','DRB1','DQB1','DQA1','DPB1','DPA1']
hla_genotype = pd.DataFrame({'patient_id': hla_per_patient['patient_id']})
for locus in loci:
    locus_cols = [c for c in four_digit if c.startswith(locus + '*')]
    sub  = hla_per_patient[locus_cols]
    mask = sub >= 1
    hla_genotype[f'HLA_{locus}'] = mask.apply(
        lambda row: ' / '.join(sub.columns[row]) if row.any() else None, axis=1)

missing = hla_genotype[[f'HLA_{l}' for l in loci]].isna().any(axis=1).sum()
print(f'Patients with complete 8-locus genotype: {len(hla_genotype) - missing}/{len(hla_genotype)}')
print(hla_genotype.head(2).to_string(index=False))

### 4. Build and save Table 1 — QC-filtered clonotypes with readable HLA genotype

Each row is one QC-passing clonotype, annotated with its patient's full HLA genotype (8 loci). This is the analysis-ready ground-truth table for any downstream TCR–HLA association work.

In [ ]:
table1 = master_clean.merge(hla_genotype, on='patient_id', how='left')

table1.to_csv(DATA + 'rosati_qc_table_with_hla.tsv.gz', sep='\t', index=False, compression='gzip')

print(f'Table 1 saved: rosati_qc_table_with_hla.tsv.gz')
print(f'Shape: {table1.shape}')
print(f'Patients: {table1["patient_id"].nunique()}')

## Part B — Validating Blind HLA Inference Against HIBAG Ground Truth

### 5. Load ground truth and prior blind-inference predictions

Each method's predictions are evaluated **independently** at a ≥90% confidence threshold, restricted to the 94 alleles modelable by HLAGuessr (the smaller, shared allele set):

- **HLAGuessr ≥90%** — from `hlaguessr_rosati_predictions.csv`
- **THNet ≥90%** — from `thnet_rosati_predictions.csv`, excluding the known DPA1*01:03 training artefact

Patient IDs are mapped between the ENA naming convention (`CD_1`) used in MiXCR-derived predictions and Elisa's naming convention (`1.CD.1.Blood.bulk`) used in the HIBAG ground truth.

In [ ]:
modelable = set(pd.read_csv(RESULTS + 'hlaguessr_46B_crossval.csv')['HLA'].unique())

hla_real = pd.read_csv(DATA + 'rosati_real_hla_long.tsv', sep='\t')
hla_real['true_carrier'] = hla_real['dosage'] >= 1
real = hla_real[hla_real['true_carrier']].groupby('patient_id')['allele']\
       .apply(lambda x: sorted(set(x))).reset_index()
real.columns = ['elisa_id', 'real_alleles']

def to_elisa_id(pid):
    num = re.search(r'(\d+)$', str(pid))
    if not num: return None
    num = num.group(1)
    if 'Healthy' in pid: return f'1.Healthy.{num}.Blood.bulk'
    if 'CD' in pid and 'UC' not in pid and 'CRC' not in pid: return f'1.CD.{num}.Blood.bulk'
    return None

print(f'Ground truth patients: {real["elisa_id"].nunique()}')
print(f'Modelable alleles: {len(modelable)}')

### 6. Validate HLAGuessr ≥90% against ground truth

In [ ]:
hlaguessr = pd.read_csv(RESULTS + 'hlaguessr_rosati_predictions.csv')
hlaguessr['patient_num'] = hlaguessr['patient_id'].str.extract(r'(\d+)$')
hlaguessr['group']       = hlaguessr['patient_id'].str.extract(r'(CD|Healthy|UC|CRC)')
hlaguessr['elisa_id']    = hlaguessr.apply(
    lambda r: f'1.Healthy.{r["patient_num"]}.Blood.bulk' if r['group']=='Healthy'
         else f'1.CD.{r["patient_num"]}.Blood.bulk' if r['group']=='CD' else None, axis=1)

hg_90 = hlaguessr[(hlaguessr['predicted_carrier']==True) &
                  (hlaguessr['probability']>=0.90) &
                  (hlaguessr['HLA'].isin(modelable))]
hg_pred = hg_90.groupby(['patient_id','elisa_id'])['HLA'].apply(lambda x: sorted(set(x))).reset_index()
hg_pred.columns = ['patient_id', 'elisa_id', 'predicted_alleles']
hg_val = hg_pred.merge(real, on='elisa_id', how='inner')

results_hg = []
for _, row in hg_val.iterrows():
    pred_set = set(row['predicted_alleles'])
    real_set = set(row['real_alleles']) & modelable
    results_hg.append({
        'patient_id': row['patient_id'], 'elisa_id': row['elisa_id'],
        'n_predicted': len(pred_set), 'n_real': len(real_set),
        'TP': len(pred_set & real_set), 'FP': len(pred_set - real_set), 'FN': len(real_set - pred_set),
        'correct': sorted(pred_set & real_set), 'wrong': sorted(pred_set - real_set),
        'missed': sorted(real_set - pred_set), 'method': 'HLAGuessr≥90%'
    })
df_hg = pd.DataFrame(results_hg)
print(f'HLAGuessr validation: {len(df_hg)} patients')

### 7. Validate THNet ≥90% against ground truth

In [ ]:
thnet = pd.read_csv(RESULTS + 'thnet_rosati_predictions.csv')
thnet = thnet[thnet['HLA'] != 'DPA1*01:03']
thnet['patient_num'] = thnet['patient_id'].str.extract(r'(\d+)$')
thnet['group']       = thnet['patient_id'].str.extract(r'(CD|Healthy|UC|CRC)')
thnet['elisa_id']    = thnet.apply(
    lambda r: f'1.Healthy.{r["patient_num"]}.Blood.bulk' if r['group']=='Healthy'
         else f'1.CD.{r["patient_num"]}.Blood.bulk' if r['group']=='CD' else None, axis=1)

thnet_90 = thnet[(thnet['predicted_carrier']==True) & (thnet['score']>=0.90) &
                 (thnet['HLA'].isin(modelable))]
thnet_pred = thnet_90.groupby(['patient_id','elisa_id'])['HLA']\
             .apply(lambda x: sorted(set(x))).reset_index()
thnet_pred.columns = ['patient_id', 'elisa_id', 'predicted_alleles']
thnet_val = thnet_pred.merge(real, on='elisa_id', how='inner')

results_thnet = []
for _, row in thnet_val.iterrows():
    pred_set = set(row['predicted_alleles'])
    real_set = set(row['real_alleles']) & modelable
    results_thnet.append({
        'patient_id': row['patient_id'], 'elisa_id': row['elisa_id'],
        'n_predicted': len(pred_set), 'n_real': len(real_set),
        'TP': len(pred_set & real_set), 'FP': len(pred_set - real_set), 'FN': len(real_set - pred_set),
        'correct': sorted(pred_set & real_set), 'wrong': sorted(pred_set - real_set),
        'missed': sorted(real_set - pred_set), 'method': 'THNet≥90%'
    })
df_thnet = pd.DataFrame(results_thnet)
print(f'THNet validation: {len(df_thnet)} patients')

### 8. Build and save Table 2 — per-patient validation and method comparison summary

In [ ]:
table2 = pd.concat([df_hg, df_thnet], ignore_index=True)

summary = table2.groupby('method').agg(
    n_patients=('patient_id','count'), total_predicted=('n_predicted','sum'),
    total_real=('n_real','sum'), TP=('TP','sum'), FP=('FP','sum'), FN=('FN','sum')
).reset_index()
summary['PPV_%']         = (summary['TP']/(summary['TP']+summary['FP'])*100).round(1)
summary['Sensitivity_%'] = (summary['TP']/(summary['TP']+summary['FN'])*100).round(1)

table2.to_csv(DATA + 'rosati_hla_validation_per_patient.tsv', sep='\t', index=False)
summary.to_csv(DATA + 'rosati_hla_validation_summary.tsv', sep='\t', index=False)

print(summary.to_string(index=False))

## 9. Conclusions

- **Quality control removed only 1.02%** of clonotypes (201,286/19,820,069), confirming Elisa Rosati's repertoires were already high quality. All 192 patients retain ≥1,300 clones per chain — well above standard depth thresholds.
- **The HIBAG-imputed HLA genotype is complete** for all 192 patients across all 8 loci, with no missing data after collapsing to 4-digit resolution.
- **Both methods, evaluated independently, achieve strong and comparable precision** against real HLA at the ≥90% confidence threshold:
  - HLAGuessr: Sensitivity 56.3%, Precision (PPV) 97.9%
  - THNet: Sensitivity 47.1%, Precision (PPV) 98.5%
  - HLAGuessr detects a somewhat larger share of true HLA alleles; THNet is marginally more precise.
- **Requiring agreement between both methods (consensus) is substantially worse than either method alone** — a separate analysis (not shown here) found consensus sensitivity of only 8.1%, confirming that dual-method agreement discards valid signal without meaningfully improving precision over single-method confidence thresholds.
- **~50% sensitivity at ~98% precision is a strong result for blind TCR-based HLA inference** — bulk repertoires were never given access to HLA labels, and public-clonotype-based inference is fundamentally limited by repertoire depth and population representation. This is not comparable to >90% benchmarks from direct molecular HLA typing (NGS/SSO), which uses genomic DNA rather than inferring genotype from immune repertoire signal.
- **Recommendation**: for downstream applications requiring high-confidence HLA calls from TCR repertoires, either method's own ≥90% threshold is reliable on its own; requiring cross-method agreement is not recommended.